# Real-Time E-Commerce Anamoly Engine
#### Author: Sachin Kumar B
#### Date: 27/06/2026

In [1]:
# Clean Up Previous Crashed State
import os
import gc

# Force garbage collection
gc.collect()

# Remove old duckdb lock files if present
for ext in ["", ".wal", ".tmp"]:
    path = f"data/notebook_lakehouse.db{ext}"
    if os.path.exists(path):
        try:
            os.remove(path)
        except PermissionError:
            pass

print("✅ Environment successfully reset!")

✅ Environment successfully reset!


In [3]:
!pip uninstall polars -y
!pip install "polars[rtcompat]"

Found existing installation: polars 1.43.0
Uninstalling polars-1.43.0:
  Successfully uninstalled polars-1.43.0
   ---------------------------------------- 0.0/846.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/846.8 kB ? eta -:--:--
   ------------------------------------- -- 786.4/846.8 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 846.8/846.8 kB 1.3 MB/s  0:00:00
   ---------------------------------------- 0.0/52.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/52.6 MB ? eta -:--:--
    --------------------------------------- 1.0/52.6 MB 2.8 MB/s eta 0:00:19
   - -------------------------------------- 1.8/52.6 MB 3.2 MB/s eta 0:00:16
   - -------------------------------------- 2.6/52.6 MB 3.5 MB/s eta 0:00:15
   -- ------------------------------------- 3.9/52.6 MB 4.0 MB/s eta 0:00:13
   --- ------------------------------------ 5.0/52.6 MB 4.2 MB/s eta 0:00:12
   ---- ----------------------------------- 6.3/52.6 MB 4.5 MB

In [4]:
import os

# Bypass strict SIMD CPU checks for Polars compatibility
os.environ["POLARS_SKIP_CPU_CHECK"] = "1"

import polars as pl
import duckdb

print(f"✅ Polars version {pl.__version__} loaded safely in compatibility mode!")

✅ Polars version 1.43.1 loaded safely in compatibility mode!


In [5]:
# Live Stream Processing & In-Notebook Dashboard
import time
import os
import random
import duckdb
import polars as pl
from datetime import datetime, timezone
from faker import Faker
from IPython.display import display, clear_output

# 1. Initialize In-Memory DuckDB (No file-locking overhead)
con = duckdb.connect(":memory:")

con.execute("""
    CREATE TABLE stream_events (
        event_id VARCHAR, user_id VARCHAR, event_type VARCHAR,
        amount DOUBLE, ip_address VARCHAR, timestamp TIMESTAMP
    )
""")

fake = Faker()
sliding_window_buffer = []

print("⚡ Streaming Anomaly Engine active (In-Memory Safe Mode).")
print("Press 'Stop' (Interrupt Kernel) to stop and export data.\n")

loop_count = 0

try:
    while True:
        loop_count += 1
        new_batch = []
        
        # --- PRODUCER STAGE ---
        for _ in range(5):
            is_anomaly = random.random() < 0.08
            event = {
                "event_id": fake.uuid4()[:8],
                "user_id": f"usr_{random.randint(100, 999)}",
                "event_type": "payment_failed" if is_anomaly else random.choice(["view", "cart_add", "checkout"]),
                "amount": round(random.uniform(500.0, 2500.0), 2) if is_anomaly else round(random.uniform(10.0, 150.0), 2),
                "ip_address": "192.168.1.1" if is_anomaly else fake.ipv4(),
                "timestamp": datetime.now(timezone.utc).isoformat()
            }
            new_batch.append(event)
            sliding_window_buffer.append(event)

        # --- CONSUMER STAGE ---
        batch_tuples = [
            (e['event_id'], e['user_id'], e['event_type'], e['amount'], e['ip_address'], e['timestamp'])
            for e in new_batch
        ]
        con.executemany("INSERT INTO stream_events VALUES (?, ?, ?, ?, ?, ?)", batch_tuples)

        # Retain last 20 events for sliding window analytics
        sliding_window_buffer = sliding_window_buffer[-20:]

        # --- ANALYTICS & DISPLAY (Throttled update) ---
        if loop_count % 3 == 0:
            df_summary = con.execute("""
                SELECT 
                    event_type, 
                    COUNT(*) as total_events, 
                    ROUND(AVG(amount), 2) as avg_spend,
                    ROUND(MAX(amount), 2) as max_spend
                FROM stream_events 
                GROUP BY event_type
            """).df()

            df_anomalies = con.execute("""
                SELECT * FROM stream_events 
                WHERE amount > 500 
                ORDER BY timestamp DESC 
                LIMIT 5
            """).df()

            clear_output(wait=True)
            print(f"⚡ REAL-TIME ANOMALY ENGINE | Batches Processed: {loop_count}")
            print("---------------------------------------------------------------------")
            print("📊 Live Stream Aggregations")
            display(df_summary)
            print("\n🚨 High-Value Anomaly Alerts (> $500)")
            display(df_anomalies)

        time.sleep(0.4)

except KeyboardInterrupt:
    print("\n🛑 Pipeline interrupted cleanly by user.")

# --- EXPORT STAGE (Runs automatically when you stop the loop) ---
print("\n💾 Exporting In-Memory Stream to Parquet Lakehouse...")

data_dir = os.path.abspath("data")
os.makedirs(data_dir, exist_ok=True)
output_parquet_dir = os.path.join(data_dir, "parquet_lakehouse").replace("\\", "/")

con.execute(f"""
    COPY stream_events TO '{output_parquet_dir}' 
    (FORMAT PARQUET, PARTITION_BY (event_type), OVERWRITE_OR_IGNORE);
""")

con.close()
print(f"✅ Export complete! Parquet partitions saved to:\n{output_parquet_dir}")

⚡ REAL-TIME ANOMALY ENGINE | Batches Processed: 537
---------------------------------------------------------------------
📊 Live Stream Aggregations


,event_type,total_events,avg_spend,max_spend
0,cart_add,830,80.79,149.81
1,view,773,77.64,149.94
2,checkout,840,81.88,149.99
3,payment_failed,242,1476.28,2492.18



🚨 High-Value Anomaly Alerts (> $500)


,event_id,user_id,event_type,amount,ip_address,timestamp
0,b36bf997,usr_375,payment_failed,702.69,192.168.1.1,2026-07-28 05:22:22.849106
1,a50679ca,usr_293,payment_failed,2137.85,192.168.1.1,2026-07-28 05:22:22.849058
2,37f527e6,usr_885,payment_failed,625.64,192.168.1.1,2026-07-28 05:22:21.952936
3,c025429f,usr_694,payment_failed,2160.96,192.168.1.1,2026-07-28 05:22:21.537385
4,ecca23c5,usr_503,payment_failed,856.93,192.168.1.1,2026-07-28 05:22:21.119189



🛑 Pipeline interrupted cleanly by user.

💾 Exporting In-Memory Stream to Parquet Lakehouse...
✅ Export complete! Parquet partitions saved to:
D:/Projects/Jupyter_Labs/Real-Time E-Commerce Anamoly Engine/data/parquet_lakehouse


In [6]:
# Verify the generated Lakehouse partition
import polars as pl

# Query the partitioned Parquet files directly using wildcard matching
df_lakehouse = pl.scan_parquet("data/parquet_lakehouse/*/*.parquet").collect()

print(f"📦 Total Parquet Records Loaded: {len(df_lakehouse)}")
print(df_lakehouse.head(5))

📦 Total Parquet Records Loaded: 2690
shape: (5, 5)
┌──────────┬─────────┬────────┬────────────────┬────────────────────────────┐
│ event_id ┆ user_id ┆ amount ┆ ip_address     ┆ timestamp                  │
│ ---      ┆ ---     ┆ ---    ┆ ---            ┆ ---                        │
│ str      ┆ str     ┆ f64    ┆ str            ┆ datetime[μs]               │
╞══════════╪═════════╪════════╪════════════════╪════════════════════════════╡
│ 192aed1b ┆ usr_979 ┆ 99.2   ┆ 31.59.24.105   ┆ 2026-07-28 05:17:46.903768 │
│ 5439dcd3 ┆ usr_820 ┆ 133.46 ┆ 75.64.209.178  ┆ 2026-07-28 05:17:46.914841 │
│ 7836cae2 ┆ usr_693 ┆ 135.83 ┆ 23.104.128.177 ┆ 2026-07-28 05:17:46.915192 │
│ ad77ad30 ┆ usr_531 ┆ 26.88  ┆ 183.140.113.72 ┆ 2026-07-28 05:17:46.915429 │
│ 2ad19473 ┆ usr_449 ┆ 67.33  ┆ 181.109.28.250 ┆ 2026-07-28 05:18:25.625968 │
└──────────┴─────────┴────────┴────────────────┴────────────────────────────┘
